In [0]:
%run "./00_config"

# Core Functions – Metadata Driven JSON Loader

All functions live here. `02_main` just calls `run_pipeline()`

Run `%run "./00_config"` first (done automatically from `02_main`)

## 1. Small helper

In [0]:
def get_full_table_name(catalog, schema, table):
    """Build catalog.schema.table as a single string."""
    return f"{catalog}.{schema}.{table}"

## 2. source_control table

The table you maintain by hand one row per source folder. Flip
`is_active` to 0 to pause a source without deleting it. Columns match
exactly what you gave us, nothing added.

| column | meaning |
|---|---|
| s_no | serial number |
| source | s3/adls/volume/dbfs |
| file_path | folder to scan (must end with `/`) |
| file_type | json |
| tgt_catalog / tgt_schema / tgt_table | where files from this source get loaded - all files under one source share one target table |
| is_active | 1 = pipeline processes this source, 0 = skipped |

In [0]:
def get_active_sources():
    ctl_full = get_full_table_name(meta_catalog, meta_schema, source_control_table)
    df = spark.table(ctl_full).filter(col("is_active") == 1)
    return [row.asDict() for row in df.collect()]

## 3. load_logs table

There's no separate file-status table. The log table itself is checked on
every run to work out which files already loaded successfully — a file's
**most recent** SUCCESS or FAILED log row is its current status.

| column | meaning |
|---|---|
| log_id | unique id for the log row |
| run_id | run id |
| executed_by | executor name |
| s_no | serial number |
| source / file_path / tgt_catalog / tgt_schema / tgt_table | target delta path|
| file_name | file name |
| status | SUCCESS / FAILED |
| start_time / end_time | timing |
| rows_processed | row count|
| message | detail, including error messages |

In [0]:
def create_log_table():
    """Create load_logs if it doesn't exist yet."""
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {get_full_table_name(meta_catalog, meta_schema, log_table)} (
            log_id          STRING,
            run_id          STRING,
            executed_by     STRING,
            s_no            INT,
            source          STRING,
            file_path       STRING,
            tgt_catalog     STRING,
            tgt_schema      STRING,
            tgt_table       STRING,
            file_name       STRING,
            status          STRING,
            start_time      TIMESTAMP,
            end_time        TIMESTAMP,
            rows_processed  BIGINT,
            message         STRING
        ) USING DELTA
    """)


LOG_SCHEMA = StructType([
    StructField("log_id", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("executed_by", StringType(), True),
    StructField("s_no", IntegerType(), True),
    StructField("source", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("tgt_catalog", StringType(), True),
    StructField("tgt_schema", StringType(), True),
    StructField("tgt_table", StringType(), True),
    StructField("file_name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("rows_processed", LongType(), True),
    StructField("message", StringType(), True),
])


def write_log(status, source_row=None, file_name=None, rows_processed=None, message=None, start_time=None):

    now = datetime.now()
    sr = source_row or {}
    row = [(
        str(uuid.uuid4()), run_id, executed_by,
        sr.get("s_no"), sr.get("source"), sr.get("file_path"),
        sr.get("tgt_catalog"), sr.get("tgt_schema"), sr.get("tgt_table"),
        file_name, status, start_time or now, now, rows_processed, message,
    )]
    df_log = spark.createDataFrame(row, schema=LOG_SCHEMA)
    df_log.write.format("delta").mode("append") \
        .saveAsTable(get_full_table_name(meta_catalog, meta_schema, log_table))

## 4. File discovery and deciding what to process

`discover_files` lists every file currently sitting in a source's folder.

`get_files_to_process` then looks at the log table for that source, finds
each file's most recent SUCCESS/FAILED result, and returns:
- files that were never logged before (brand new files), plus
- files whose last attempt was FAILED

Files whose last attempt was SUCCESS are left out, so a re-run only
touches what's new or previously failed.

In [0]:
def discover_files(source_row):
    """List every file in a source's folder matching its file_type."""
    files = dbutils.fs.ls(source_row["file_path"])
    ext = "." + source_row["file_type"]
    return [f.name for f in files if f.name.endswith(ext)]


def get_latest_file_status(s_no):
    """For a given source, return {file_name: last_status} using only the
    most recent SUCCESS/FAILED log row per file."""
    log_full = get_full_table_name(meta_catalog, meta_schema, log_table)
    df = (
        spark.table(log_full)
             .filter(col("s_no") == s_no)
             .filter(col("status").isin("SUCCESS", "FAILED"))
             .filter(col("file_name").isNotNull())
    )
    window = Window.partitionBy("file_name").orderBy(desc("end_time"))
    latest = (
        df.withColumn("rn", row_number().over(window))
          .filter(col("rn") == 1)
          .select("file_name", "status")
    )
    return {row.file_name: row.status for row in latest.collect()}


def get_files_to_process(source_row, discovered_files):
    """Return the files that still need loading: new files, or files whose
    last logged attempt was FAILED. Files last logged as SUCCESS are skipped."""
    latest_status = get_latest_file_status(source_row["s_no"])
    return [f for f in discovered_files if latest_status.get(f) != "SUCCESS"]

## 4b. extract_schema 
This function recursively extracts the schema from a Spark DataFrame and returns all field names as a flat Python list, including nested fields. It is especially useful when working with JSON data containing nested objects (StructType) and arrays (ArrayType).

In [0]:
def extract_schema(schema, parent=""):
    fields = []

    for field in schema.fields:
        name = f"{parent}.{field.name}" if parent else field.name
        dtype = field.dataType

        if isinstance(dtype, StructType):
            fields.append(name)
            fields.extend(extract_schema(dtype, name))

        elif isinstance(dtype, ArrayType):
            fields.append(f"{name}[]")

            if isinstance(dtype.elementType, StructType):
                fields.extend(extract_schema(dtype.elementType, f"{name}[]"))

        else:
            fields.append(name)

    return fields

## 5. Loading a single file

Reads one file and appends it into the source's target table (all files
from the same source share one table, as set in source_control).

File names follow the pattern
`carereport_hsar_<id>_<period_start>_<period_end>_<dt_data_file>_<HHMMSS>`.
`dt_data_file` is the date the file was generated/extracted (the token
right before the trailing time-of-day) and is pulled out with
`extract_dt_data_file` below.

In [0]:
def extract_dt_data_file(file_name):
    """Pull dt_data_file out of a file name.

    Example: carereport_hsar_5105283582043366812_20220101_20251106_20251113_102927.json
    -> "20251113" (the date token right before the trailing HHMMSS time token).

    Returns None if the file name doesn't have enough underscore-separated parts.
    """
    stem = file_name.rsplit(".", 1)[0]  # drop extension, if any
    parts = stem.split("_")
    return parts[-2] if len(parts) >= 2 else None

In [0]:
def process_file(source_row, file_name):
    file_type = source_row["file_type"].lower()
    file_full_path = source_row["file_path"] + file_name
    tgt_full = get_full_table_name(source_row["tgt_catalog"], source_row["tgt_schema"], source_row["tgt_table"])

    file_info = dbutils.fs.ls(file_full_path)[0]
    file_size = file_info.size
    file_modified_date = datetime.fromtimestamp(file_info.modificationTime / 1000)
    dt_data_file = extract_dt_data_file(file_name)

    if file_type == "json":
        df_schema = (
            spark.read.option("multiline", "true").json(file_full_path)
        )

        df = (
            spark.read
                 .option("wholetext", True)
                 .text(file_full_path)
                 .select(
                     lit(source_row["file_path"]).alias("file_path"),
                     lit(file_name).alias("file_name"),
                     col("value").alias("json_original_data"),
                     lit(file_size).alias("file_size"),
                     lit(file_modified_date).alias("file_modified_date"),
                     lit(dt_data_file).alias("dt_data_file"),
                     current_timestamp().alias("loaded_at"),
                 )
        )

        schema_list = extract_schema(df_schema.schema)

        df_final = df.withColumn(
            "file_schema",
            array(*[lit(col_name) for col_name in schema_list])
            ).select(
            "file_path",
            "file_name",
            "json_original_data", # Original JSON
            parse_json("json_original_data").alias("json_data_for_parsing"),
            "file_schema",
            "file_size",
            "file_modified_date",
            "dt_data_file",
            "loaded_at",
        )
    else:
        raise NotImplementedError(f"file_type '{file_type}' is not supported yet")

    row_count = df_final.count()
    df_final.write.format("delta").mode("append").saveAsTable(tgt_full)
    return row_count

## 6. Orchestration

The single entry point `02_main` calls. For every active source: list all
files in the folder, work out which ones still need loading (new or
previously failed), and load them one by one — one file failing doesn't
stop the others, and every outcome is logged.

In [0]:
def run_pipeline():

    create_log_table()

    print("Pipeline run started")

    active_sources = get_active_sources()
    print(f"{len(active_sources)} active source(s) found in source_control")

    total_success = 0
    total_fail = 0

    for source_row in active_sources:
        discovered = discover_files(source_row)
        print(f"Source {source_row['s_no']}: found {len(discovered)} file(s) in folder")

        files_to_process = get_files_to_process(source_row, discovered)
        print(f"Source {source_row['s_no']}: {len(files_to_process)} file(s) are new or previously failed, will be processed")

        for file_name in files_to_process:
            start_time = datetime.now()
            try:
                rows = process_file(source_row, file_name)
                write_log(
                    status="SUCCESS", source_row=source_row, file_name=file_name,
                    rows_processed=rows, start_time=start_time,
                    message="File loaded successfully",
                )
                total_success += 1
            except Exception as e:
                error_message = str(e)[:500]
                write_log(
                    status="FAILED", source_row=source_row, file_name=file_name,
                    start_time=start_time, message=error_message,
                )
                total_fail += 1

    print(f"Pipeline finished. Success: {total_success}, Failed: {total_fail}")